# Chapter 4 — Fitting five dwarf galaxies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/chemical-evolution-tutorial/blob/main/notebooks/04_fitting.ipynb)

**Goal.** Fit H21's model (Chapter 3) to the [Si/Fe]–[Fe/H] tracks of the five galaxies (Chapter 0), using a grid of
3403 precomputed models, the same grid H21 searched. We define the score H21 used, check on mock stars that it
recovers a model we know, find the best fits, and compare them with H21's published ones. This chapter needs no
VICE: the grid was computed with the code of Chapter 3, and Chapter 3 checked five of its models.

**You will learn**
1. how to measure the distance between stars and a model track;
2. why every fit must first be tested on mock data with a known answer;
3. what the best fits are, which parameters are degenerate, and how well H21's results reproduce;
4. why a χ² that is far above 1 per star should be read as a ranking, not as a probability.

**Run time.** Scoring the full grid takes about 2–4 minutes.

In [ ]:
import hashlib, json, urllib.request, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# --- Data access (identical in every chapter) -------------------------------------------------------------
DATA_URL = 'https://github.com/vasilybelokurov/chemical-evolution-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def fetch(name, cache=Path('data')):
    """Return the local path of data file `name`, downloading it if needed; its SHA-256 must match the manifest."""
    want = MANIFEST['files'][name]['sha256']
    path = Path('../data_build') / name
    if not path.exists():
        path = cache / name
        if not path.exists() or sha256(path) != want:
            cache.mkdir(exist_ok=True)
            print(f'downloading {name} ...')
            urllib.request.urlretrieve(DATA_URL + name, path)
    assert sha256(path) == want, f'{name}: checksum mismatch'
    return path


GALAXIES = {'LMC': 'lmc', 'SMC': 'smc', 'Sgr': 'sgr', 'GSE': 'gse', 'Fornax': 'fornax'}
COLORS = {'LMC': '#2a78d6', 'SMC': '#eb6834', 'Sgr': '#1baf7a', 'GSE': '#eda100', 'Fornax': '#e87ba4'}
KEYS = ('sfe0', 'eta', 'tau_b', 'sigma_b', 'f_b')


def good(t, x):
    """The tutorial's quality cut (Chapter 0)."""
    ok = np.ones(len(t), bool)
    for c in ('feh', x):
        ok &= (t[f'{c}_flag'] == 0) & np.isfinite(t[c]) & (t[c] > -9000) & (t[f'{c}_err'] > 0)
    return ok


stars = {}
for g, k in GALAXIES.items():
    t = Table.read(fetch(f'{k}_h21.ecsv'))
    t = t[good(t, 'sife')]
    stars[g] = (np.c_[t['feh'], t['sife']].astype(float), np.c_[t['feh_err'], t['sife_err']].astype(float))

G = np.load(fetch('h21_grid_vice.npz'))          # 3403 models: parameters and tracks on a 0.03-Gyr time grid
print({g: int(np.sum(G['galaxy'] == g)) for g in GALAXIES}, 'models;', len(G['time']), 'time steps')

# H21 Table 4: best-fit parameters (None: not used; GSE has no burst)
H21_BEST = {
    'LMC':    dict(sfe0=0.015, eta=2.0,   tau_b=11.0, sigma_b=0.75, f_b=6.0),
    'SMC':    dict(sfe0=0.008, eta=10.0,  tau_b=7.0,  sigma_b=0.75, f_b=4.0),
    'Sgr':    dict(sfe0=0.03,  eta=17.5,  tau_b=5.0,  sigma_b=0.5,  f_b=0.01),
    'GSE':    dict(sfe0=0.14,  eta=6.0,   tau_b=None, sigma_b=None, f_b=1.0),
    'Fornax': dict(sfe0=0.03,  eta=100.0, tau_b=6.0,  sigma_b=5.0,  f_b=0.01),
}

## 4.1 The score

A model is a *track*: a curve in the ([Fe/H], [Si/Fe]) plane. H21 score a track by how far each star lies from it.
For a star at $p$ with errors $(\sigma_{\rm Fe}, \sigma_{\rm Si})$, find the nearest point $q$ on the track and the
residual $d = p - q$. The star's error *in the direction of* $d$ is
$\sigma_d^2 = (d_{\rm Fe}^2\sigma_{\rm Fe}^2 + d_{\rm Si}^2\sigma_{\rm Si}^2)/|d|^2$, so its contribution is

$$\chi^2_i = \frac{|d|^2}{\sigma_d^2} = \frac{|d|^4}{d_{\rm Fe}^2\sigma_{\rm Fe}^2 + d_{\rm Si}^2\sigma_{\rm Si}^2}.$$

The track is a sequence of points joined by straight segments; the nearest point can lie anywhere on a segment.

In [ ]:
def chi2_track(points, errors, track):
    """Sum over stars of |d|^4 / sum_k d_k^2 sigma_k^2, d = star minus nearest point on the polyline `track`.
    points, errors: (N, 2) arrays of ([Fe/H], [Si/Fe]) and their errors; track: (M, 2). Returns chi2, N."""
    v = np.all(np.isfinite(track), axis=1)
    seg = v[:-1] & v[1:]                                     # segments whose two ends are finite
    a, b = track[:-1][seg], track[1:][seg]
    ab = b - a
    ab2 = np.einsum('ij,ij->i', ab, ab)                      # squared segment lengths
    chi = np.empty(len(points))
    for s in range(0, len(points), 128):                     # chunks of 128 stars keep the arrays small
        p, e = points[s:s + 128], errors[s:s + 128]
        d = p[:, None, :] - a                                # each star minus each segment start: (n, m, 2)
        f = np.clip(np.divide(np.einsum('nmi,mi->nm', d, ab), ab2, out=np.zeros(d.shape[:2]), where=ab2 > 0), 0, 1)
        q = a + f[:, :, None] * ab                           # nearest point on each segment
        d2 = np.sum((p[:, None, :] - q) ** 2, axis=2)
        k = np.argmin(d2, axis=1)                            # nearest segment of each star
        r = p - q[np.arange(len(k)), k]                      # residual vector d
        den = np.sum(r ** 2 * e ** 2, axis=1)
        chi[s:s + 128] = np.divide(d2[np.arange(len(k)), k] ** 2, den, out=np.zeros(len(k)), where=den > 0)
    return float(chi.sum()), len(points)


# A hand check: a star 0.1 dex above a horizontal track has d = (0, 0.1), so chi2 = 0.1^2 / sigma_Si^2.
c, n = chi2_track(np.array([[-1.0, 0.1]]), np.array([[0.1, 0.05]]), np.array([[-2.0, 0.0], [0.0, 0.0]]))
assert abs(c - 0.1 ** 2 / 0.05 ** 2) < 1e-12                 # identity: = 4
# ... and a star beyond the end of the track is measured from the end point: d = (0.3, 0.4), |d| = 0.5
c, n = chi2_track(np.array([[0.3, 0.4]]), np.array([[0.1, 0.1]]), np.array([[-2.0, 0.0], [0.0, 0.0]]))
assert abs(c - 0.5 ** 4 / (0.3 ** 2 * 0.01 + 0.4 ** 2 * 0.01)) < 1e-12   # identity: = 25
print('hand checks pass')

**The end-point term.** A track that runs far past the most metal-rich stars costs nothing in this score: no star
is near its end. H21 added a penalty for such tracks but did not give its formula. We use the reconstruction of
the `cerium_bump` project: the "end point" of the data is the most metal-rich star's [Fe/H] together with the
median [Si/Fe] of the 30 most metal-rich stars; every model step beyond that [Fe/H] adds a term like the one above,
measured from the end point. Steps in the last 1 Gyr of the model are exempt, and GSE (not a satellite, whose star
formation ended long ago) is never penalised. Keep in mind that this term is our guess at H21's.

In [ ]:
def chi2_endpoint(points, errors, track, time, galaxy, dt=0.03, duration=12.0):
    """Penalty for model steps beyond the most metal-rich star (reconstruction of H21's term; not for GSE)."""
    if galaxy == 'GSE':
        return 0.0
    top = np.argsort(points[:, 0], kind='stable')[-30:]      # the 30 most metal-rich stars
    end = np.array([points[:, 0].max(), np.median(points[top, 1])])
    err = np.median(errors[top], axis=0)
    use = np.all(np.isfinite(track), axis=1) & (track[:, 0] > end[0]) & (time <= duration - 1 + 1e-9)
    d = track[use] - end
    d2, den = np.sum(d ** 2, axis=1), np.sum(d ** 2 * err ** 2, axis=1)
    return float(np.sum(np.divide(d2 * d2, den, out=np.zeros_like(d2), where=den > 0)) * dt / 0.03)


def score(galaxy, pts, err, rows):
    """Total score (track + end point) of grid models `rows` of one galaxy against stars pts, err."""
    out = np.empty(len(rows))
    for j, i in enumerate(rows):
        track = np.c_[G['feh'][i], G['sife'][i]].astype(float)
        out[j] = chi2_track(pts, err, track)[0] + chi2_endpoint(pts, err, track, G['time'], galaxy)
    return out

## 4.2 Test the fit on mock stars first

Before fitting real stars, fit fake ones made from a model we choose, and check that the fit finds it. We use GSE:
its grid has two parameters (SFE$_0$ and $\eta$; 143 models) and no end-point term. The mock is made the way nature
makes stars:
- pick a birth time for each star with probability proportional to the model's star-formation rate;
- put the star on the model track at that time;
- add Gaussian noise with the errors of a real GSE star (drawn at random from the 1811).

In [ ]:
rows_gse = np.flatnonzero(G['galaxy'] == 'GSE')
truth = rows_gse[np.flatnonzero(np.isclose(G['sfe0'][rows_gse], 0.14) & np.isclose(G['eta'][rows_gse], 6.0))[0]]


def mock_stars(i, errors, rng):
    """Mock stars from grid model i: birth times weighted by SFR, Gaussian noise with errors drawn from `errors`."""
    ok = np.isfinite(G['feh'][i]) & (G['time'] > 0)
    w = (G['sfr'][i] * ok) / np.sum(G['sfr'][i] * ok)        # probability of being born at each time step
    birth = rng.choice(len(w), size=len(errors), p=w)
    e = errors[rng.integers(0, len(errors), len(errors))]   # errors of randomly chosen real stars
    p = np.c_[G['feh'][i][birth], G['sife'][i][birth]] + rng.normal(size=e.shape) * e
    return p, e


rng = np.random.default_rng(42)
t0 = time.time()
recovered, chi_true = [], []
for trial in range(10):
    p, e = mock_stars(truth, stars['GSE'][1], rng)
    s = score('GSE', p, e, rows_gse)
    recovered.append(rows_gse[np.argmin(s)])
    chi_true.append(s[rows_gse == truth][0] / len(p))
recovered = np.array(recovered)
hits = np.mean(recovered == truth)
print(f'10 mocks of GSE (truth SFE0 = 0.14, eta = 6): truth recovered in {hits:.0%}; '
      f'chi2/N of the true model: {np.mean(chi_true):.2f} ± {np.std(chi_true):.2f}   ({time.time() - t0:.0f} s)')
print('recovered (SFE0, eta):', sorted({(float(G["sfe0"][i]), float(G["eta"][i])) for i in recovered}))
assert hits >= 0.8                                           # recovery: the fit finds the input model
assert abs(np.mean(chi_true) - 1) < 0.1                      # with correct errors, chi2/N of the truth is ~1

The fit recovers the input model in at least 8 of 10 mocks (printed above), and when it misses, it lands on a grid
neighbour. With the right errors, the true model has $\chi^2/N \approx 1$, as it should. This is the baseline
against which to read the real fits.

## 4.3 The real fits

In [ ]:
t0 = time.time()
scores = {}
for g in GALAXIES:
    rows = np.flatnonzero(G['galaxy'] == g)
    scores[g] = (rows, score(g, *stars[g], rows))
print(f'scored {sum(len(r) for r, _ in scores.values())} models in {time.time() - t0:.0f} s')


def params(i):
    return {k: float(G[k][i]) for k in KEYS if np.isfinite(G[k][i])}


def is_h21(g, i):
    return all(np.isnan(G[k][i]) if H21_BEST[g][k] is None else np.isclose(G[k][i], H21_BEST[g][k]) for k in KEYS)


summary = {}
for g, (rows, s) in scores.items():
    n = len(stars[g][0])
    order = np.argsort(s)
    best = rows[order[0]]
    h = int(np.flatnonzero([is_h21(g, rows[j]) for j in order])[0])       # position of H21's model in the ranking
    summary[g] = dict(best=best, chi2n=s.min() / n, rank=h + 1, scaled_dchi2=(s[order[h]] - s.min()) / (s.min() / n))
    print(f"{g:7s} N = {n:4d}  best {params(best)}  chi2/N = {s.min() / n:.2f};  "
          f"H21's model ranks {h + 1}/{len(rows)}, scaled Δχ² = {summary[g]['scaled_dchi2']:.1f}")

# reproduction: H21's best fit is (nearly) ours for Sgr, GSE and the LMC ...
assert summary['Sgr']['rank'] == 1 and summary['GSE']['rank'] == 1 and summary['LMC']['rank'] <= 3
# ... and is not for the SMC and Fornax (regression: frozen at the development run, 2026-11-12)
assert summary['SMC']['rank'] == 30 and summary['Fornax']['rank'] == 304
fx = G['feh'][summary['Fornax']['best']]
print(f"Fornax best track ends at [Fe/H] = {fx[-1]:.2f}; most metal-rich star: {stars['Fornax'][0][:, 0].max():.2f}")
assert fx[-1] > stars['Fornax'][0][:, 0].max() + 0.2         # the loophole discussed below Figure 4.1

**Reading the numbers.** The best fits have $\chi^2/N$ = 4.4–7.1, not ~1 as in the mocks. The quoted APOGEE
errors do not include the real scatter of stars about a single track: a galaxy is not one perfectly mixed zone,
and abundance errors have systematic parts. So $\chi^2$ differences cannot be turned into probabilities directly.
A common fix, used here, is to scale the errors up until the best model has $\chi^2/N = 1$; a difference in
scaled $\Delta\chi^2$ of 1 (4) is then roughly a 1σ (2σ) difference for one parameter. It is a rough guide, not a
confidence interval.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 7))
for j, g in enumerate(GALAXIES):
    rows, s = scores[g]
    pts, err = stars[g]
    h21 = rows[np.flatnonzero([is_h21(g, i) for i in rows])[0]]
    ax = axes[0, j]
    ax.scatter(pts[:, 0], pts[:, 1], s=3 if len(pts) > 500 else 8, color=COLORS[g], alpha=0.3, lw=0)
    ax.plot(G['feh'][h21], G['sife'][h21], color='#d62828', lw=1.8, label='H21 Table 4')
    ax.plot(G['feh'][summary[g]['best']], G['sife'][summary[g]['best']], color='k', lw=1.8, ls='--', label='best fit here')
    ax.set(xlim=(-2.5, 0.1), ylim=(-0.5, 0.6), xlabel='[Fe/H]', title=g)
    ax.legend(frameon=False, fontsize=8, loc='lower left')
    ax = axes[1, j]
    ax.plot(G['time'], G['sfr'][h21], color='#d62828', lw=1.8)
    ax.plot(G['time'], G['sfr'][summary[g]['best']], color='k', lw=1.8, ls='--')
    ax.set(xlabel='time [Gyr]', xlim=(0, 12))
axes[0, 0].set_ylabel('[Si/Fe]')
axes[1, 0].set_ylabel(r'SFR [$M_\odot\,$yr$^{-1}$]')
for ax in axes.flat:
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

**Figure 4.1.** Top: the stars of each galaxy (points) with H21's best model (red, Table 4) and the best model of our
grid search (black dashed), in [Si/Fe] against [Fe/H]. Bottom: the star-formation rates of the two models against
time.

For **Sgr** and **GSE** the two are the same model. For the **LMC** ours has a stronger outflow ($\eta$ = 4 instead
of 2) and a stronger burst ($f_b$ = 8 instead of 6), but the tracks are nearly indistinguishable. For the **SMC**,
our fit prefers a stronger outflow and a later, stronger burst, and H21's model ranks 30th. For **Fornax**, H21's
model ranks 304th of 1260. Our best Fornax track has its dip at higher [Fe/H] and then runs to [Fe/H] = −0.15, far
past the most metal-rich star: that excursion happens in the model's last Gyr, which the end-point term exempts. A
fit can exploit any loophole in its score; this is one.

## 4.4 Which parameters are constrained?

Profiles show this: for each value of one parameter, the best score over all the others.

In [ ]:
grid_keys = {'LMC': ('sfe0', 'eta', 'tau_b', 'f_b'), 'SMC': ('sfe0', 'eta', 'tau_b', 'f_b'),
             'Sgr': ('sfe0', 'eta', 'tau_b', 'sigma_b'), 'GSE': ('sfe0', 'eta'), 'Fornax': ('sfe0', 'eta', 'tau_b', 'sigma_b')}
fig, axes = plt.subplots(5, 4, figsize=(14, 14))
for r, g in enumerate(GALAXIES):
    rows, s = scores[g]
    scaled = (s - s.min()) / (s.min() / len(stars[g][0]))
    for c in range(4):
        ax = axes[r, c]
        if c >= len(grid_keys[g]):
            ax.axis('off')
            continue
        k = grid_keys[g][c]
        vals = np.unique(G[k][rows])
        prof = [scaled[np.isclose(G[k][rows], v)].min() for v in vals]
        ax.plot(vals, prof, 'o-', color=COLORS[g], lw=1.8, ms=4)
        ax.axvline(H21_BEST[g][k], color='#d62828', lw=1.2, alpha=0.8)
        for y in (1, 4):
            ax.axhline(y, color='k', lw=0.6, ls=':')
        ax.set(yscale='symlog', ylim=(-0.1, 1e3), xlabel=k)
        ax.grid(color='0.9', lw=0.6)
        if c == 0:
            ax.set_ylabel(f'{g}\nscaled Δχ²')
fig.tight_layout()

rows, s = scores['Fornax']
scaled = (s - s.min()) / (s.min() / len(stars['Fornax'][0]))
prof_at_h21 = {k: scaled[np.isclose(G[k][rows], H21_BEST['Fornax'][k])].min() for k in grid_keys['Fornax']}
print('Fornax: profile at H21 values', {k: round(float(v), 1) for k, v in prof_at_h21.items()})
assert max(prof_at_h21.values()) < 3 and summary['Fornax']['scaled_dchi2'] > 100   # each fine alone, not together

**Figure 4.2.** Profiles of the scaled $\Delta\chi^2$ (errors scaled so the best model has $\chi^2/N$ = 1; symmetric-log
scale) against each grid parameter, minimised over the others. Rows: galaxies; red lines: H21's Table 4 values;
dotted lines: $\Delta\chi^2$ = 1 and 4.

The efficiency SFE$_0$ is the best-constrained parameter in every galaxy: it sets the knee (Chapter 2). The burst
time $\tau_b$ is sharply constrained in the LMC and Sgr, where the burst leaves a clear mark on the track. The outflow
$\eta$ is constrained in GSE, Sgr and the SMC. Two best values sit at the edge of the grid: the LMC's burst
strength ($f_b$ = 8, the largest searched) and, nearly, Fornax's outflow ($\eta$ = 100, one step below the largest);
such values are limits, not measurements. For Fornax, every parameter on its own is within scaled $\Delta\chi^2
\le$ 2 of H21's value, yet H21's *combination* ranks 304th with scaled $\Delta\chi^2$ = 209: the parameters
are correlated, and only certain combinations fit.

## 4.5 Is it the code?

H21 used the flexCE code; we used VICE. Could the SMC and Fornax disagreements come from the code? The tutorial's
data include the scores of the same 3403 models run with flexCE (H21's own code, with the same yields), computed
outside this notebook with an earlier quality cut that differs from ours by at most 12 stars per galaxy.

In [ ]:
S = Table.read(fetch('h21_grid_scores.csv'), format='ascii.csv')
corr = {}
for g in GALAXIES:
    v = np.asarray(S['chi2'][(S['engine'] == 'vice') & (S['galaxy'] == g)], float)
    f = np.asarray(S['chi2'][(S['engine'] == 'flexce') & (S['galaxy'] == g)], float)
    rv, rf = np.argsort(np.argsort(v)), np.argsort(np.argsort(f))           # ranks of each model in each code
    corr[g] = np.corrcoef(rv, rf)[0, 1]                                     # Spearman rank correlation
    print(f'{g:7s} rank correlation VICE vs flexCE over {len(v)} models: {corr[g]:.3f};  '
          f'median |Δχ²|/χ² = {np.median(np.abs(f - v) / v):.3f}')
assert min(corr.values()) > 0.98                             # the two codes rank the models the same way

The two codes rank the models almost identically (rank correlation 0.99–1.00) and their scores differ by 1–3%.
The disagreement with H21 for the SMC and Fornax is not a code difference. It must come from what goes *into* the
fit: the member stars (Chapter 0 showed GSE's membership does not even reproduce H21's count), the quality cut, the
end-point term we had to reconstruct, or details of H21's fitting we do not know. Agreement between two codes rules
out the code; it does not tell us which of the inputs is responsible.

## Summary

- H21's score is each star's distance to the track in units of its error along the residual; we checked it by hand
  and recovered a known model from mock stars.
- Best fits have $\chi^2/N$ = 4.4–7.1: real stars scatter more than their quoted errors about any one-zone track. Use
  $\chi^2$ to rank models, and scale it before reading differences as uncertainties.
- H21's results reproduce for Sgr, GSE and the LMC, not for the SMC and Fornax; VICE and flexCE agree, so the code
  is not the cause.
- Star-formation efficiency is well constrained everywhere; outflows and bursts only in some galaxies.

## Exercises

1. Make 10 mocks from the SMC's H21 model with the SMC's errors, and fit them (this needs the end-point term). How
   often is the input recovered?
2. Remove the end-point term and refit Sgr. Where does the best fit go, and why?
3. Drop the 46 Fornax stars below [Fe/H] = −1.1 (Chapter 0) and refit Fornax. How does H21's rank change?